# 03 · Árboles de decisión y bagging desde cero

**Módulo 4 · Sesión 10** — Árboles de decisión y bagging

## Objetivos

Los tres clasificadores de la sesión 9 —logística, KNN, SVM— comparten una idea: una
frontera definida por distancias o productos punto sobre variables escaladas. Los árboles
rompen con eso: parten el espacio con preguntas de la forma "¿$x_j \leq t$?", una a la vez.
Este notebook construye la idea desde cero:

1. Medir la **impureza** de un nodo (Gini y entropía) y encontrar la mejor partición a
   mano, sobre una variable y luego sobre todas.
2. Construir un árbol **recursivo** en ~40 líneas y verificar que coincide con
   `DecisionTreeClassifier`.
3. Ver cómo la profundidad controla el sobreajuste — la misma curva en U de
   `05-sesgo-varianza-validacion.md`, con otra perilla — y qué hace la poda.
4. Medir la **inestabilidad** de un árbol: cuánto cambia si cambian unos pocos datos.
5. Implementar **bagging** a mano y medir cuánta varianza elimina; y ver por qué
   Random Forest añade aleatoriedad en las variables, no solo en las filas.

La teoría está en `04-arboles-y-bagging.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scikit-learn`.

In [ ]:
# Arranque para Google Colab (en local no hace nada): trae el repositorio para que
# ../datos y ../src existan. Ejecútala antes que cualquier otra celda.
import sys
if "google.colab" in sys.modules:
    !git clone -q --depth 1 https://github.com/delany-ramirez/machine_learning /content/machine_learning
    %cd /content/machine_learning/modulo-4-clasificacion-ensambles/notebooks

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_moons
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos: dos medias lunas con ruido

Un problema sintético en 2D, no separable linealmente, con ruido suficiente para que
memorizar sea una mala idea. Sirve para **ver** las fronteras que cada método construye.

In [ ]:
X, y = make_moons(n_samples=600, noise=0.32, random_state=SEMILLA)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=SEMILLA)


def graficar_frontera(modelo_predict, X, y, eje, titulo):
    g1, g2 = np.meshgrid(np.linspace(-2, 3, 300), np.linspace(-1.75, 2.25, 300))
    z = modelo_predict(np.c_[g1.ravel(), g2.ravel()]).reshape(g1.shape)
    eje.contourf(g1, g2, z, levels=[-0.5, 0.5, 1.5], colors=["C0", "C1"], alpha=0.25)
    eje.scatter(X[y == 0, 0], X[y == 0, 1], c="C0", s=10)
    eje.scatter(X[y == 1, 0], X[y == 1, 1], c="C1", s=10)
    eje.set_title(titulo)
    eje.set_xticks([])
    eje.set_yticks([])


fig, eje = plt.subplots(figsize=(6, 4.5))
eje.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c="C0", s=12, label="clase 0")
eje.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c="C1", s=12, label="clase 1")
eje.set_title("Datos de entrenamiento (n = 300)")
eje.legend()
plt.show()

## 2. Impureza: qué tan "mezclado" está un nodo

Un árbol parte los datos en regiones y predice, en cada una, la clase mayoritaria. Para
decidir dónde partir necesita una medida de cuánto se mezclan las clases en un conjunto de
puntos. Con $p_k$ la fracción de la clase $k$ en el nodo:

$$
\text{Gini} = 1 - \sum_k p_k^2 \qquad\qquad \text{Entropía} = -\sum_k p_k \log_2 p_k
$$

Ambas valen 0 cuando el nodo es puro (una sola clase) y son máximas cuando las clases están
a partes iguales. En la práctica dan casi siempre la misma partición.

In [ ]:
def gini(y):
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - np.sum(p**2)


def entropia(y):
    p = np.bincount(y, minlength=2) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


p1 = np.linspace(0.001, 0.999, 200)
plt.figure(figsize=(6, 4))
plt.plot(p1, 1 - p1**2 - (1 - p1) ** 2, label="Gini")
plt.plot(p1, -(p1 * np.log2(p1) + (1 - p1) * np.log2(1 - p1)), label="Entropía")
plt.plot(p1, 1 - np.maximum(p1, 1 - p1), "--", color="gray", label="Error de clasificación")
plt.xlabel("Fracción de la clase 1 en el nodo")
plt.ylabel("Impureza")
plt.legend()
plt.title("Las tres medidas de impureza para dos clases")
plt.show()

print(f"Gini del nodo raíz (entrenamiento): {gini(y_train):.3f}   entropía: {entropia(y_train):.3f}")

> El error de clasificación también sirve como impureza, pero es **lineal por tramos**: una
> partición que lleva un nodo de 50/50 a dos nodos de 70/30 no lo reduce nada, aunque
> claramente avanzó. Gini y entropía son estrictamente cóncavas y sí premian ese avance.
> Por eso los árboles crecen con Gini o entropía y se **evalúan** con accuracy.

### La mejor partición de una variable

Una partición "$x_j \leq t$" divide el nodo en izquierda y derecha. Su calidad es la
**reducción de impureza**, ponderando cada lado por su tamaño:

$$
\Delta = \text{Gini}(\text{nodo}) - \frac{n_I}{n}\text{Gini}(I) - \frac{n_D}{n}\text{Gini}(D)
$$

Como el criterio solo depende de qué puntos caen a cada lado, basta probar los umbrales
entre valores consecutivos de $x_j$ ordenados. Es una búsqueda exhaustiva, y por eso es
rápida: $O(n \log n)$ por variable.

In [ ]:
def mejor_particion_variable(x, y):
    """Mejor umbral para una variable: devuelve (reducción de impureza, umbral)."""
    orden = np.argsort(x)
    x_ord, y_ord = x[orden], y[orden]
    n = len(y)
    impureza_nodo = gini(y)
    mejor = (0.0, None)
    for i in range(1, n):
        if x_ord[i] == x_ord[i - 1]:
            continue  # no se puede partir entre dos valores iguales
        izq, der = y_ord[:i], y_ord[i:]
        delta = impureza_nodo - (i / n) * gini(izq) - ((n - i) / n) * gini(der)
        if delta > mejor[0]:
            mejor = (delta, (x_ord[i] + x_ord[i - 1]) / 2)
    return mejor


fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
for j, eje in enumerate(ejes):
    orden = np.argsort(X_train[:, j])
    xs = X_train[orden, j]
    deltas = []
    for i in range(1, len(xs)):
        izq, der = y_train[orden][:i], y_train[orden][i:]
        deltas.append(gini(y_train) - (i / len(xs)) * gini(izq) - ((len(xs) - i) / len(xs)) * gini(der))
    delta, umbral = mejor_particion_variable(X_train[:, j], y_train)
    eje.plot(xs[1:], deltas)
    eje.axvline(umbral, color="C3", ls="--", label=f"mejor: x{j+1} ≤ {umbral:.2f} (Δ = {delta:.3f})")
    eje.set_xlabel(f"umbral sobre x{j+1}")
    eje.set_ylabel("Reducción de Gini")
    eje.legend()
plt.tight_layout()
plt.show()

La variable $x_2$ ofrece la mayor reducción: la primera pregunta del árbol será
"$x_2 \leq$ ese umbral". Ahora, el árbol completo: aplicar la misma búsqueda sobre todas las
variables, partir, y **repetir recursivamente** en cada hijo hasta una profundidad máxima o
hasta que el nodo sea puro.

In [ ]:
def construir_arbol(X, y, profundidad_max, profundidad=0, min_muestras=2):
    """Árbol binario como diccionarios anidados. Hoja = {'clase': k}."""
    clase_mayoritaria = int(np.bincount(y, minlength=2).argmax())
    if profundidad == profundidad_max or len(y) < min_muestras or gini(y) == 0:
        return {"clase": clase_mayoritaria, "n": len(y)}

    mejor_delta, mejor_j, mejor_t = 0.0, None, None
    for j in range(X.shape[1]):
        delta, t = mejor_particion_variable(X[:, j], y)
        if delta > mejor_delta:
            mejor_delta, mejor_j, mejor_t = delta, j, t
    if mejor_j is None:
        return {"clase": clase_mayoritaria, "n": len(y)}

    mascara = X[:, mejor_j] <= mejor_t
    return {
        "variable": mejor_j,
        "umbral": mejor_t,
        "n": len(y),
        "izq": construir_arbol(X[mascara], y[mascara], profundidad_max, profundidad + 1, min_muestras),
        "der": construir_arbol(X[~mascara], y[~mascara], profundidad_max, profundidad + 1, min_muestras),
    }


def predecir_uno(arbol, x):
    while "clase" not in arbol:
        arbol = arbol["izq"] if x[arbol["variable"]] <= arbol["umbral"] else arbol["der"]
    return arbol["clase"]


def predecir(arbol, X):
    return np.array([predecir_uno(arbol, x) for x in X])


def imprimir(arbol, sangria=""):
    if "clase" in arbol:
        print(f"{sangria}→ clase {arbol['clase']} (n={arbol['n']})")
        return
    print(f"{sangria}x{arbol['variable']+1} ≤ {arbol['umbral']:.3f}  (n={arbol['n']})")
    imprimir(arbol["izq"], sangria + "  ├ sí: ")
    imprimir(arbol["der"], sangria + "  └ no: ")


arbol_mano = construir_arbol(X_train, y_train, profundidad_max=3)
imprimir(arbol_mano)

### Comparación con `DecisionTreeClassifier`

In [ ]:
arbol_sk = DecisionTreeClassifier(max_depth=3, random_state=SEMILLA).fit(X_train, y_train)

print("Raíz de scikit-learn:", f"x{arbol_sk.tree_.feature[0]+1} ≤ {arbol_sk.tree_.threshold[0]:.3f}")
print(f"Predicciones idénticas en prueba: {np.mean(predecir(arbol_mano, X_test) == arbol_sk.predict(X_test)):.3f}")
print(f"Accuracy en prueba — a mano: {np.mean(predecir(arbol_mano, X_test) == y_test):.3f}   "
      f"scikit-learn: {arbol_sk.score(X_test, y_test):.3f}")

fig, eje = plt.subplots(figsize=(13, 5))
plot_tree(arbol_sk, feature_names=["x1", "x2"], class_names=["0", "1"], filled=True, impurity=True, ax=eje, fontsize=9)
plt.show()

Mismas particiones, mismas predicciones. Lo que `scikit-learn` añade es velocidad
(Cython), criterios adicionales y las opciones de poda de la sección siguiente. Fíjese en la
gráfica del árbol: cada nodo reporta su Gini y el conteo por clase — es la misma
`gini()` de arriba.

## 3. Profundidad, sobreajuste y poda

Sin límite, el árbol sigue partiendo hasta que cada hoja es pura: memoriza el conjunto de
entrenamiento, ruido incluido. La profundidad es la perilla sesgo-varianza del árbol.

In [ ]:
profundidades = range(1, 16)
acc_train, acc_test = [], []
for d in profundidades:
    a = DecisionTreeClassifier(max_depth=d, random_state=SEMILLA).fit(X_train, y_train)
    acc_train.append(a.score(X_train, y_train))
    acc_test.append(a.score(X_test, y_test))

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for eje, d in zip(ejes[:3], [1, 3, None]):
    a = DecisionTreeClassifier(max_depth=d, random_state=SEMILLA).fit(X_train, y_train)
    graficar_frontera(a.predict, X_train, y_train, eje, f"profundidad {d or 'sin límite'} (hojas: {a.get_n_leaves()})")
ejes[3].plot(profundidades, acc_train, "o-", label="entrenamiento")
ejes[3].plot(profundidades, acc_test, "o-", label="prueba")
ejes[3].set_xlabel("max_depth")
ejes[3].set_ylabel("accuracy")
ejes[3].set_title("Curva de validación")
ejes[3].legend()
plt.tight_layout()
plt.show()

mejor_d = list(profundidades)[int(np.argmax(acc_test))]
print(f"Accuracy en prueba: profundidad 1 = {acc_test[0]:.3f}, mejor ({mejor_d}) = {max(acc_test):.3f}, "
      f"sin límite = {acc_test[-1]:.3f}")

El árbol sin límite alcanza accuracy 1.0 en entrenamiento y 0.867 en prueba, dos puntos
por debajo del mejor (profundidad 2, 0.887): las regiones diminutas de la tercera gráfica
son puntos de ruido memorizados. Y el tocón de profundidad 1 (0.810) se queda corto: una
sola pregunta no puede describir dos medias lunas. Las fronteras son siempre **rectángulos
alineados con los ejes** — un árbol no puede dibujar una diagonal, la aproxima con
escalones.

### Poda por costo-complejidad

Fijar `max_depth` es una poda **a priori**. La alternativa clásica (Breiman, 1984) es dejar
crecer el árbol completo y luego podar **a posteriori** las ramas que no compensan su
complejidad, minimizando

$$
R_\alpha(T) = R(T) + \alpha \cdot |\text{hojas}(T)|
$$

donde $R(T)$ es la impureza total de las hojas. Cada $\alpha$ define un subárbol; se elige
el $\alpha$ por validación. Es Lasso para árboles: $\alpha$ penaliza el número de hojas como
$\lambda$ penalizaba la norma de los coeficientes.

In [ ]:
ruta = DecisionTreeClassifier(random_state=SEMILLA).cost_complexity_pruning_path(X_train, y_train)
alphas = ruta.ccp_alphas[:-1]  # el último alpha deja un solo nodo

hojas, acc_test_poda = [], []
for a in alphas:
    arbol = DecisionTreeClassifier(ccp_alpha=a, random_state=SEMILLA).fit(X_train, y_train)
    hojas.append(arbol.get_n_leaves())
    acc_test_poda.append(arbol.score(X_test, y_test))

fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))
ejes[0].plot(alphas, hojas, "o-", ms=3)
ejes[0].set_xscale("log")
ejes[0].set_xlabel(r"$\alpha$")
ejes[0].set_ylabel("número de hojas")
ejes[1].plot(alphas, acc_test_poda, "o-", ms=3)
ejes[1].set_xscale("log")
ejes[1].set_xlabel(r"$\alpha$")
ejes[1].set_ylabel("accuracy en prueba")
plt.tight_layout()
plt.show()

i = int(np.argmax(acc_test_poda))
print(f"Mejor alpha: {alphas[i]:.4f} → {hojas[i]} hojas, accuracy en prueba {acc_test_poda[i]:.3f} "
      f"(sin poda: {hojas[0]} hojas, {acc_test_poda[0]:.3f})")

## 4. La inestabilidad del árbol

Un árbol tiene **alta varianza** en un sentido muy concreto: cambiar unos pocos puntos de
entrenamiento puede cambiar la primera partición y, con ella, todo el árbol. Lo medimos
con **bootstrap** —remuestrear las filas con reemplazo, como en
`03-regularizacion-intuicion.ipynb` (módulo 3) para los coeficientes de OLS— y observamos
cuánto cambia la frontera.

In [ ]:
def muestra_bootstrap(n, rng):
    return rng.integers(0, n, size=n)


fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for k, eje in enumerate(ejes):
    idx = muestra_bootstrap(len(y_train), rng)
    a = DecisionTreeClassifier(max_depth=None, random_state=SEMILLA).fit(X_train[idx], y_train[idx])
    graficar_frontera(a.predict, X_train, y_train, eje, f"remuestra bootstrap {k+1}")
plt.suptitle("El mismo algoritmo, cuatro remuestras de los mismos datos", y=1.02)
plt.tight_layout()
plt.show()

Cuatro fronteras muy distintas para el mismo problema. Los errores de cada árbol están en
lugares **diferentes** — y eso es justo lo que se puede explotar.

## 5. Bagging: promediar árboles inestables

**Bootstrap aggregating**: entrenar $B$ árboles sobre $B$ remuestras bootstrap y
**promediar** sus predicciones (votar, en clasificación; promediar las probabilidades es
mejor). Si los árboles fueran independientes, la varianza del promedio sería
$\sigma^2 / B$. No lo son —comparten la mayoría de los datos—, pero la reducción es grande
de todos modos. Y el sesgo no cambia: el promedio de árboles profundos sigue siendo un
modelo flexible.

En ~10 líneas:

In [ ]:
def bagging_ajustar(X, y, B, rng, profundidad=None):
    arboles = []
    for _ in range(B):
        idx = muestra_bootstrap(len(y), rng)
        arboles.append(DecisionTreeClassifier(max_depth=profundidad, random_state=SEMILLA).fit(X[idx], y[idx]))
    return arboles


def bagging_proba(arboles, X):
    return np.mean([a.predict_proba(X)[:, 1] for a in arboles], axis=0)


Bs = [1, 2, 5, 10, 25, 50, 100, 200]
acc_por_B = []
for B in Bs:
    arboles = bagging_ajustar(X_train, y_train, B, np.random.default_rng(SEMILLA))
    acc_por_B.append(np.mean((bagging_proba(arboles, X_test) >= 0.5) == y_test))

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.6))
for eje, B in zip(ejes[:3], [1, 10, 200]):
    arboles = bagging_ajustar(X_train, y_train, B, np.random.default_rng(SEMILLA))
    graficar_frontera(lambda Z: (bagging_proba(arboles, Z) >= 0.5).astype(int), X_train, y_train, eje, f"bagging, B = {B}")
ejes[3].plot(Bs, acc_por_B, "o-")
ejes[3].axhline(max(acc_test), color="gray", ls="--", label=f"mejor árbol solo ({max(acc_test):.3f})")
ejes[3].set_xscale("log")
ejes[3].set_xlabel("B (número de árboles)")
ejes[3].set_ylabel("accuracy en prueba")
ejes[3].legend(fontsize=8)
plt.tight_layout()
plt.show()

print(pd.Series(acc_por_B, index=Bs, name="accuracy en prueba").round(3).to_frame().T)

Con $B = 1$ es un árbol sin podar sobre una remuestra (0.837). Al promediar, la frontera se
suaviza y la accuracy sube hasta estabilizarse en ≈0.88 desde $B \approx 5$ — el nivel del
mejor árbol podado de la sección 3, **sin haber elegido ninguna profundidad**. Bagging **no
sobreajusta con $B$**: más árboles nunca empeoran, solo cuestan más tiempo. Que aquí empate
con el mejor árbol y no lo supere es propio de un problema de 2 variables con mucho ruido:
el techo lo pone el ruido, no el modelo. La ganancia sobre el árbol solo se ve mejor cuando
el árbol tiene más de qué sobreajustar (sección 6).

### Midiendo la varianza directamente

La afirmación "bagging reduce la varianza" se puede comprobar: repetimos el experimento
completo 30 veces con distintas semillas y medimos cuánto varía la probabilidad predicha
para cada punto de prueba entre repeticiones, para un árbol solo y para bagging.

In [ ]:
def varianza_predicciones(ajustar_y_predecir, repeticiones=30):
    """Desviación estándar, entre repeticiones, de P(clase 1) en cada punto de prueba."""
    predicciones = np.array([ajustar_y_predecir(np.random.default_rng(s)) for s in range(repeticiones)])
    return predicciones.std(axis=0).mean()


def un_arbol(rng):
    idx = muestra_bootstrap(len(y_train), rng)
    return DecisionTreeClassifier(random_state=SEMILLA).fit(X_train[idx], y_train[idx]).predict_proba(X_test)[:, 1]


def bagging_50(rng):
    return bagging_proba(bagging_ajustar(X_train, y_train, 50, rng), X_test)


print(f"Desviación típica de P(clase 1) entre repeticiones — árbol solo: {varianza_predicciones(un_arbol):.3f}   "
      f"bagging (B=50): {varianza_predicciones(bagging_50):.3f}")

## 6. Random Forest: decorrelacionar los árboles

El límite de bagging es que sus árboles se **parecen**: si una variable es claramente la
mejor para la raíz, casi todas las remuestras la eligen, y los árboles quedan
correlacionados. La varianza de un promedio de $B$ variables con correlación $\rho$ es

$$
\rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2
$$

El segundo término se va con $B$; el primero **no**. Random Forest ataca $\rho$: en cada
nodo, en vez de buscar la mejor partición entre todas las variables, la busca entre un
subconjunto aleatorio de `max_features` de ellas. Árboles individualmente peores, pero
menos parecidos entre sí — y el promedio gana.

Con solo 2 variables no hay nada que muestrear. Añadimos 18 variables de ruido puro —lo
habitual en datos reales es tener muchas variables de las que pocas importan— y
comparamos árbol solo, bagging y Random Forest con dos valores de `max_features`.

In [ ]:
# 2 variables reales + 18 columnas de ruido gaussiano puro
Xr_train = np.hstack([X_train, rng.normal(size=(len(X_train), 18))])
Xr_test = np.hstack([X_test, rng.normal(size=(len(X_test), 18))])

comparacion = {}
for nombre, modelo in {
    "árbol solo": DecisionTreeClassifier(random_state=SEMILLA),
    "bagging (B=200, todas las variables)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=SEMILLA),
    "Random Forest (B=200, max_features='sqrt')": RandomForestClassifier(n_estimators=200, random_state=SEMILLA),
    "Random Forest (B=200, max_features=1)": RandomForestClassifier(n_estimators=200, max_features=1, random_state=SEMILLA),
}.items():
    modelo.fit(Xr_train, y_train)
    comparacion[nombre] = modelo.score(Xr_test, y_test)

print("Accuracy en prueba con 2 variables reales + 18 de ruido:")
print(pd.Series(comparacion).round(3).to_string())

> `BaggingClassifier` con `DecisionTreeClassifier` es exactamente el bagging de la sección
> 5; `RandomForestClassifier` es lo mismo más el muestreo de variables en cada nodo
> (`max_features='sqrt'` por defecto: $\sqrt{20} \approx 4$ de las 20).

Dos resultados, uno esperado y otro no. El esperado: el árbol solo se hunde con ruido
(0.77) y bagging lo rescata por completo (0.89). El inesperado: **Random Forest no supera a
bagging aquí**, y con `max_features=1` es claramente peor. La razón es aritmética: de 20
variables solo 2 sirven, y un subconjunto de 4 al azar no contiene **ninguna** de las dos en
una fracción grande de los nodos —esos nodos parten por ruido—:

In [ ]:
for m in [1, 4, 10, 20]:
    p_sin_reales = comb(18, m) / comb(20, m) if m <= 18 else 0.0
    print(f"max_features = {m:>2}: probabilidad de que el subconjunto no incluya ninguna variable real = {p_sin_reales:.2f}")

Decorrelacionar árboles ayuda cuando **muchas** variables llevan señal y compiten por la
raíz; cuando casi todas son ruido, restringir la elección hace árboles peores sin
hacerlos menos parecidos en lo que importa. `max_features` es un hiperparámetro que se
afina, no una mejora automática. El notebook 04 mide bagging contra Random Forest sobre
Wine Quality, donde las 11 variables fisicoquímicas sí llevan señal.

### Error *out-of-bag*: validación gratis

Cada remuestra bootstrap deja fuera, en promedio, el $(1 - 1/n)^n \approx 37\,\%$ de las
filas. Esas filas son un conjunto de validación natural para ese árbol. Promediando, para
cada fila, solo los árboles que **no** la vieron, se obtiene una estimación del error de
generalización sin partir los datos ni hacer validación cruzada.

In [ ]:
n = 300
fuera_simulada = np.mean([1 - len(set(muestra_bootstrap(n, rng))) / n for _ in range(200)])
print(f"Fracción de filas fuera de una remuestra bootstrap de n={n}: teórica {(1 - 1/n)**n:.3f}, "
      f"simulada {fuera_simulada:.3f}")

rf = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=SEMILLA).fit(Xr_train, y_train)
print(f"Accuracy OOB: {rf.oob_score_:.3f}   accuracy en prueba: {rf.score(Xr_test, y_test):.3f}")

## Resumen

| Concepto | Lo que se vio |
|---|---|
| Impureza (Gini, entropía) | Cóncavas; el error de clasificación no sirve para crecer el árbol |
| Mejor partición | Búsqueda exhaustiva sobre umbrales ordenados; la recursión da el árbol |
| Profundidad | Perilla sesgo-varianza; sin límite, accuracy 1.0 en entrenamiento y 0.867 en prueba frente a 0.887 del mejor |
| Poda por costo-complejidad | Penaliza el número de hojas, como Lasso penaliza coeficientes |
| Inestabilidad | Cuatro remuestras bootstrap, cuatro fronteras distintas |
| Bagging | Promedia árboles profundos; la desviación típica de las predicciones baja de 0.139 a 0.023; no sobreajusta con $B$ |
| Random Forest | Bagging + variables aleatorias por nodo, para bajar la correlación $\rho$; con 2 variables útiles de 20, **no** gana a bagging |
| OOB | El 37 % que cada remuestra deja fuera valida ese árbol gratis |

Todo lo anterior son fronteras en 2D con datos sintéticos. El notebook 04 aplica árboles,
Random Forest y boosting sobre Wine Quality, y mide si superan la AP ≈ 0.53 en la que se
quedaron la logística, KNN y la SVM.